# Fig. 06 Disentanglement Comparison with LLM

Self-contained notebook version of the Figure 6 renderer. It loads the submitted visual and LLM interactions and writes the direct exports to `Fig_06_fit/outputs/`.


In [1]:
from pathlib import Path
import sys
import math
from functools import lru_cache

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.ticker import MaxNLocator

PACKAGE_ROOT = next((path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / "interactions").is_dir()), None)
if PACKAGE_ROOT is None:
    raise FileNotFoundError("Run this notebook from inside AIJ_fig_submit.")
FIGURE_DIR = PACKAGE_ROOT / "Fig_06_fit"
SCRIPT_DIR = FIGURE_DIR / "source"
sys.path.insert(0, str(SCRIPT_DIR))
from row_cache import input_signature, load as load_row_cache, save as save_row_cache


## Shared Helpers


In [2]:
STYLE = {
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.unicode_minus": False,
    "axes.edgecolor": "#555555",
    "axes.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "xtick.major.size": 3.0,
    "ytick.major.size": 3.0,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.alpha": 0.22,
    "grid.linestyle": "--",
    "grid.linewidth": 0.45,
    "grid.color": "#BBBBBB",
    "figure.facecolor": "white",
    "axes.facecolor": "#FAFAFA",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
}


def apply_style(base_font_size=9):
    cfg = dict(STYLE)
    cfg["font.size"] = base_font_size
    cfg["legend.fontsize"] = max(base_font_size - 1, 6)
    cfg["xtick.labelsize"] = max(base_font_size - 1, 6)
    cfg["ytick.labelsize"] = max(base_font_size - 1, 6)
    plt.rcParams.update(cfg)


def plot_signed_bars(ax, pos, neg, pos_color, neg_color, width=0.78, show_zero=True):
    x = np.arange(1, len(pos) + 1)
    ax.bar(x, pos, width=width, color=pos_color, edgecolor="none")
    ax.bar(x, neg, width=width, color=neg_color, edgecolor="none")
    if show_zero:
        ax.axhline(0, color="#333333", linewidth=0.7)
    ax.set_xlim(0.4, len(pos) + 0.6)
    ax.set_xticks([1, len(pos)] if len(pos) > 4 else x)
    ax.tick_params(pad=1)


def generate_all_masks(length):
    masks = [np.binary_repr(mask, width=length) for mask in range(2**length)]
    return [[bool(int(item)) for item in mask] for mask in masks]


def list_harsanyi_samples(root, prefix=None):
    root = Path(root)
    if prefix:
        samples = []
        for cluster in sorted(p for p in root.iterdir() if p.is_dir() and p.name.startswith(prefix)):
            samples.extend(cluster.name + "/" + sample.name for sample in sorted(cluster.iterdir()) if sample.is_dir())
        return samples
    return [p.name for p in sorted(root.iterdir()) if p.is_dir()]


def load_thresholded_interactions(root, samples, tau):
    root = Path(root)
    I_and_list = []
    I_or_list = []
    vn_v0 = []
    masks = None
    for sample_name in samples:
        sample_dir = root / sample_name
        I_and_list.append(np.load(sample_dir / "after_sparsify" / "Iand.npy"))
        I_or_list.append(np.load(sample_dir / "after_sparsify" / "Ior.npy"))
        rewards = np.load(sample_dir / "before_sparsify" / "rewards.npy")
        vn_v0.append(abs(float(rewards[0] - rewards[-1])))
        if masks is None:
            masks = np.load(sample_dir / "before_sparsify" / "masks.npy")

    I_and = np.asarray(I_and_list, dtype=float)
    I_or = np.asarray(I_or_list, dtype=float)
    threshold = float(np.mean(vn_v0) * tau)
    I_and[np.abs(I_and) < threshold] = 0.0
    I_or[np.abs(I_or) < threshold] = 0.0
    return I_and, I_or, masks


def aggregate_signed_orders(I_and, I_or, masks):
    n_dim = masks.shape[1]
    orders = np.sum(masks, axis=1).astype(int)
    pos = np.maximum(I_and, 0) + np.maximum(I_or, 0)
    neg = np.minimum(I_and, 0) + np.minimum(I_or, 0)
    pos_mean = np.mean(pos, axis=0)
    neg_mean = np.mean(neg, axis=0)
    pos_order = np.zeros(n_dim)
    neg_order = np.zeros(n_dim)
    for order in range(1, n_dim + 1):
        idx = orders == order
        pos_order[order - 1] = np.sum(pos_mean[idx])
        neg_order[order - 1] = np.sum(neg_mean[idx])
    return pos_order, neg_order


def shared_interactions(target_I_and, target_I_or, baseline_I_and_list, baseline_I_or_list):
    baseline_I_and = np.asarray(baseline_I_and_list, dtype=float)
    baseline_I_or = np.asarray(baseline_I_or_list, dtype=float)

    target_pos_and = np.maximum(target_I_and, 0)
    target_neg_and = -np.minimum(target_I_and, 0)
    target_pos_or = np.maximum(target_I_or, 0)
    target_neg_or = -np.minimum(target_I_or, 0)

    baseline_pos_and = np.maximum(np.max(baseline_I_and, axis=0), 0)
    baseline_neg_and = -np.minimum(np.min(baseline_I_and, axis=0), 0)
    baseline_pos_or = np.maximum(np.max(baseline_I_or, axis=0), 0)
    baseline_neg_or = -np.minimum(np.min(baseline_I_or, axis=0), 0)

    shared_and = np.where(baseline_pos_and == 0, 0, target_pos_and) - np.where(baseline_neg_and == 0, 0, target_neg_and)
    shared_or = np.where(baseline_pos_or == 0, 0, target_pos_or) - np.where(baseline_neg_or == 0, 0, target_neg_or)
    return shared_and, shared_or


def load_target_and_shared_interactions(target_root, baseline_template, baseline_seeds, prefix=None, tau=0.02):
    samples = list_harsanyi_samples(target_root, prefix)
    target_I_and, target_I_or, masks = load_thresholded_interactions(target_root, samples, tau)
    baseline_I_and_list = []
    baseline_I_or_list = []
    for seed in baseline_seeds:
        baseline_root = str(baseline_template).replace("{seed}", str(seed))
        baseline_I_and, baseline_I_or, _ = load_thresholded_interactions(baseline_root, samples, tau)
        baseline_I_and_list.append(baseline_I_and)
        baseline_I_or_list.append(baseline_I_or)
    shared_I_and, shared_I_or = shared_interactions(target_I_and, target_I_or, baseline_I_and_list, baseline_I_or_list)
    return target_I_and, target_I_or, shared_I_and, shared_I_or, masks

## Decay / Spindle Fitting


In [3]:
def _generate_subset_masks_torch(set_mask, all_masks):
    import torch

    set_mask_ = set_mask.expand_as(all_masks)
    is_subset = torch.logical_or(set_mask_, torch.logical_not(all_masks))
    return torch.all(is_subset, dim=1)


def _generate_intersection_masks_torch(set_mask, all_masks):
    import torch

    set_mask_ = set_mask.expand_as(all_masks)
    have_intersection = torch.logical_and(set_mask_, all_masks)
    return torch.any(have_intersection, dim=1)


def triggering_mat_and(dim):
    import torch

    all_masks = torch.BoolTensor(generate_all_masks(dim))
    rows = []
    for mask_s in all_masks:
        row = torch.zeros(len(all_masks), dtype=torch.float64)
        row[_generate_subset_masks_torch(mask_s, all_masks)] = 1.0
        rows.append(row)
    return torch.stack(rows).double()


def triggering_mat_or(dim):
    import torch

    all_masks = torch.BoolTensor(generate_all_masks(dim))
    rows = []
    empty_mask = torch.zeros(dim).bool()
    empty_index = _generate_subset_masks_torch(empty_mask, all_masks)
    for mask_s in all_masks:
        row = torch.zeros(len(all_masks), dtype=torch.float64)
        row[empty_index] = 1.0
        row[_generate_intersection_masks_torch(mask_s, all_masks)] = 1.0
        rows.append(row)
    return torch.stack(rows).double()


def M_mat_and(dim, sigma2):
    import torch

    J = triggering_mat_and(dim)
    JtJ = torch.mm(J.t(), J)
    masks = torch.BoolTensor(generate_all_masks(dim))
    masks_sum = torch.sum(masks, dim=1, dtype=torch.float64)
    c = torch.pow(2, masks_sum) * float(sigma2)
    diag_c = torch.diag(c) * 2**dim
    return torch.linalg.inv(JtJ + diag_c) @ JtJ


def M_mat_or(dim, sigma2):
    import torch

    J = triggering_mat_or(dim)
    JtJ = torch.mm(J.t(), J)
    masks = torch.BoolTensor(generate_all_masks(dim))
    masks_sum = torch.sum(masks, dim=1, dtype=torch.float64)
    c = torch.pow(2, masks_sum) * float(sigma2)
    diag_c = torch.diag(c) * 2**dim
    return torch.linalg.inv(JtJ + diag_c) @ JtJ


def _general_comb(alpha, order):
    alpha = float(alpha)
    if alpha < order:
        return 0.0
    value = 1.0
    for i in range(order):
        value *= alpha - i
    return max(value / math.factorial(order), 0.0)


def compute_noise_interaction_distribution(alpha, beta, order):
    import torch

    alpha_order = order * float(alpha)
    values = [_general_comb(alpha_order, j) for j in range(1, order + 1)]
    return torch.tensor(values, dtype=torch.float64) * float(beta)


def _fit_decay_spindle(
    target_pos, target_neg, noise_metric,
    reliable_pos_metric, reliable_neg_metric,
    alpha_list, beta_list, sigma2_list, gamma_list,
):
    """Grid search for alpha, beta, sigma2, gamma that best fits
    target = decay + spindle. Returns best params and component arrays."""
    import torch

    reliable_pos_flat = reliable_pos_metric.reshape(-1, reliable_pos_metric.shape[-1])
    reliable_neg_flat = reliable_neg_metric.reshape(-1, reliable_neg_metric.shape[-1])
    n_gamma = len(gamma_list)

    min_loss = None
    alpha_idx = beta_idx = reliable_idx = 0
    chunk_size = 16
    for start in range(0, len(reliable_pos_flat), chunk_size):
        end = min(start + chunk_size, len(reliable_pos_flat))
        pos = noise_metric.unsqueeze(2) + reliable_pos_flat[start:end].unsqueeze(0).unsqueeze(0)
        neg = noise_metric.unsqueeze(2) + reliable_neg_flat[start:end].unsqueeze(0).unsqueeze(0)
        loss = torch.linalg.norm(pos - target_pos, dim=-1, ord=2) + torch.linalg.norm(
            neg - target_neg, dim=-1, ord=2
        )
        chunk_min = loss.min()
        if min_loss is None or chunk_min < min_loss:
            min_loss = chunk_min
            idx = torch.where(loss == chunk_min)
            alpha_idx = int(idx[0][0])
            beta_idx = int(idx[1][0])
            reliable_idx = start + int(idx[2][0])

    sigma_idx = reliable_idx // n_gamma
    gamma_idx = reliable_idx % n_gamma

    alpha = alpha_list[alpha_idx]
    beta = beta_list[beta_idx]
    sigma2 = sigma2_list[sigma_idx]
    gamma = gamma_list[gamma_idx]

    spindle_pos = compute_noise_interaction_distribution(alpha, beta, len(target_pos))
    spindle_neg = spindle_pos.clone()
    decay_pos = reliable_pos_metric[sigma_idx, gamma_idx]
    decay_neg = reliable_neg_metric[sigma_idx, gamma_idx]

    return {
        "all_pos": target_pos.numpy(),
        "all_neg": target_neg.numpy(),
        "decay_pos": decay_pos.numpy(),
        "decay_neg": decay_neg.numpy(),
        "spindle_pos": spindle_pos.numpy(),
        "spindle_neg": spindle_neg.numpy(),
        "alpha": float(alpha),
        "beta": float(beta),
        "gamma": float(gamma),
        "sigma2": float(sigma2),
        "min_loss": float(min_loss),
    }


def disentangle_interaction_orders(
    I_and,
    I_or,
    masks,
    *,
    cache_dir,
    tau=0.03,
    set_predict_tau=None,
    alpha_list=None,
    beta_list=None,
    sigma2_list=None,
    gamma_list=None,
):
    """Fit target interactions as reliable(decay) + noise(spindle).

    Caches the reliable interaction distribution per sigma2 under ``cache_dir``
    so that recomputation is skipped on subsequent runs.
    """
    import torch

    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)

    n_dim = masks.shape[1]
    threshold_tau = set_predict_tau if set_predict_tau is not None else tau

    if alpha_list is None:
        alpha_list = torch.linspace(0, 200, 201, dtype=torch.float64) * 1e-2
    if beta_list is None:
        beta_list = torch.linspace(0, 500, 1001, dtype=torch.float64) * 1e-3
    if sigma2_list is None:
        sigma2_list = torch.tensor([1e-3, 1e-2, 1e-1, 1.0], dtype=torch.float64)
    if gamma_list is None:
        gamma_list = torch.linspace(0, 10, 1001, dtype=torch.float64)

    target_pos, target_neg = _signed_order_tensors(I_and, I_or, masks)

    # Precompute noise (spindle) metric over alpha x beta.
    noise_metric = torch.zeros(len(alpha_list), len(beta_list), n_dim, dtype=torch.float64)
    for i, alpha in enumerate(alpha_list):
        base = compute_noise_interaction_distribution(alpha, 1.0, n_dim)
        noise_metric[i] = base.unsqueeze(0) * beta_list.unsqueeze(1)

    # Precompute reliable (decay) metric over sigma2 x gamma, with caching.
    reliable_pos_metric = torch.zeros(len(sigma2_list), len(gamma_list), n_dim, dtype=torch.float64)
    reliable_neg_metric = torch.zeros(len(sigma2_list), len(gamma_list), n_dim, dtype=torch.float64)
    for sigma_i, sigma2 in enumerate(sigma2_list):
        pos_cache = cache_dir / f"pos_{sigma2}.out"
        neg_cache = cache_dir / f"neg_{sigma2}.out"
        if pos_cache.exists() and neg_cache.exists():
            base_pos = torch.load(pos_cache, weights_only=True)
            base_neg = torch.load(neg_cache, weights_only=True)
        else:
            base_pos, base_neg = _reliable_order_tensors(
                I_and, I_or, masks, float(sigma2), tau=threshold_tau
            )
            torch.save(base_pos, pos_cache)
            torch.save(base_neg, neg_cache)
        reliable_pos_metric[sigma_i] = base_pos.unsqueeze(0) * gamma_list.unsqueeze(1)
        reliable_neg_metric[sigma_i] = base_neg.unsqueeze(0) * gamma_list.unsqueeze(1)

    return _fit_decay_spindle(
        target_pos, target_neg, noise_metric,
        reliable_pos_metric, reliable_neg_metric,
        alpha_list, beta_list, sigma2_list, gamma_list,
    )


def _signed_order_tensors(I_and, I_or, masks):
    import torch

    n_dim = masks.shape[1]
    orders = np.sum(masks, axis=1).astype(int)
    I_and = torch.tensor(I_and, dtype=torch.float64)
    I_or = torch.tensor(I_or, dtype=torch.float64)
    pos = torch.zeros(n_dim, dtype=torch.float64)
    neg = torch.zeros(n_dim, dtype=torch.float64)
    pos_vec = torch.where(I_and > 0, I_and, torch.tensor(0.0, dtype=torch.float64)) + torch.where(I_or > 0, I_or, torch.tensor(0.0, dtype=torch.float64))
    neg_vec = -torch.where(I_and < 0, I_and, torch.tensor(0.0, dtype=torch.float64)) - torch.where(I_or < 0, I_or, torch.tensor(0.0, dtype=torch.float64))
    pos_mean = torch.mean(pos_vec, dim=0)
    neg_mean = torch.mean(neg_vec, dim=0)
    for order in range(1, n_dim + 1):
        idx = torch.tensor(orders == order, dtype=torch.bool)
        pos[order - 1] = pos_mean[idx].sum()
        neg[order - 1] = neg_mean[idx].sum()
    return pos, neg


def _reliable_order_tensors(star_I_and, star_I_or, masks, sigma2, tau=0.02):
    import torch

    n_dim = masks.shape[1]
    orders = np.sum(masks, axis=1).astype(int)
    M_and = M_mat_and(n_dim, float(sigma2))
    M_or = M_mat_or(n_dim, float(sigma2))
    pos_total = torch.zeros(2**n_dim, dtype=torch.float64)
    neg_total = torch.zeros(2**n_dim, dtype=torch.float64)
    for I_and, I_or in zip(star_I_and, star_I_or):
        reliable_and = M_and @ torch.tensor(I_and, dtype=torch.float64)
        reliable_or = M_or @ torch.tensor(I_or, dtype=torch.float64)
        threshold = torch.abs(torch.sum(reliable_and[1:]) + torch.sum(reliable_or[1:])) * tau
        reliable_and[torch.abs(reliable_and) < threshold] = 0.0
        reliable_or[torch.abs(reliable_or) < threshold] = 0.0
        pos_total += torch.where(reliable_and > 0, reliable_and, torch.tensor(0.0, dtype=torch.float64))
        pos_total += torch.where(reliable_or > 0, reliable_or, torch.tensor(0.0, dtype=torch.float64))
        neg_total -= torch.where(reliable_and < 0, reliable_and, torch.tensor(0.0, dtype=torch.float64))
        neg_total -= torch.where(reliable_or < 0, reliable_or, torch.tensor(0.0, dtype=torch.float64))

    pos_vec = pos_total / len(star_I_and)
    neg_vec = neg_total / len(star_I_and)
    pos = torch.zeros(n_dim, dtype=torch.float64)
    neg = torch.zeros(n_dim, dtype=torch.float64)
    for order in range(1, n_dim + 1):
        idx = torch.tensor(orders == order, dtype=torch.bool)
        pos[order - 1] = pos_vec[idx].sum()
        neg[order - 1] = neg_vec[idx].sum()
    return pos, neg

## Figure Configuration and Rendering


In [4]:
OUTPUT_DIR = FIGURE_DIR / "outputs"

DATA_ROOT = str(PACKAGE_ROOT / "interactions" / "generalization")

LLM_SINGLE_CACHE_ROOT = PACKAGE_ROOT / "interactions" / "robustness_n" / "llm_target_single_cache_globalq" / "n=10"

LLM_TASK = "finance"

LLM_MASKS_PATH = PACKAGE_ROOT / "interactions" / "llm" / "masks.npy"

SEED_NUM = 3

BAR_WIDTH = 0.72

SUBPLOT_WIDTH = 1.5

SUBPLOT_HEIGHT = 0.82

WSPACE_INCHES = 0.12

GROUP_GAP_INCHES = 0.50

HSPACE_INCHES = 0.20

LEFT_MARGIN = 0.15

RIGHT_MARGIN = 1.25

TOP_MARGIN = 0.02

BOTTOM_MARGIN = 0.20

N_COLS = 5

COLORS = {
    "all_pos": "#CE593B",
    "all_neg": "#F6CDBF",
    "part_pos": "#4B73C3",
    "part_neg": "#A9C0EE",
}

MODELS = [
    {
        "key": "mnist-resnet20",
        "label": "ResNet-20 on MNIST",
        "prefix": "1_cluster",
        "tau": 0.01,
        "set_predict_tau": 0.03,
        "y_scaler": 2,
        "target_root": DATA_ROOT + "/harsanyi_spilt/split/dataset=mnist_split_model=resnet20_epoch=20_bs=128_lr=0.01_weight-decay=0.0005_seed=0_noise-label=False",
        "baseline_template": DATA_ROOT + "/harsanyi_spilt/split_baseline/dataset=mnist_split_model=resnet20_epoch=20_bs=128_lr=0.01_weight-decay=0.0005_seed={seed}_noise-label=False",
    },
    {
        "key": "cifar10-resnet20",
        "label": "ResNet-20 on CIFAR-10",
        "prefix": "class-",
        "tau": 0.03,
        "y_scaler": 2,
        "target_root": DATA_ROOT + "/harsanyi_spilt_adv/split/dataset=cifar10_split_model=resnet20_epoch=200_bs=128_lr=0.01_weight-decay=0.0005_seed=0_noise-label=False_step-size=4.0_epsilon=64.0_n-step=1",
        "baseline_template": DATA_ROOT + "/harsanyi_spilt_adv/split_baseline/dataset=cifar10_split_model=resnet20_epoch=200_bs=128_lr=0.01_weight-decay=0.0005_seed={seed}_noise-label=False_step-size=4.0_epsilon=64.0_n-step=1",
    },
    {
        "key": "cifar10-vgg11",
        "label": "VGG-11 on CIFAR-10",
        "prefix": "class-",
        "tau": 0.03,
        "y_scaler": 2,
        "target_root": DATA_ROOT + "/harsanyi_spilt_adv/split/dataset=cifar10_split_model=vgg11_epoch=80_bs=128_lr=0.01_weight-decay=0.0005_seed=0_noise-label=False_step-size=4.0_epsilon=64.0_n-step=1",
        "baseline_template": DATA_ROOT + "/harsanyi_spilt_adv/split_baseline/dataset=cifar10_split_model=vgg11_epoch=80_bs=128_lr=0.01_weight-decay=0.0005_seed={seed}_noise-label=False_step-size=4.0_epsilon=64.0_n-step=1",
    },
    {
        "key": "sst2-bert",
        "label": "BERT-Medium on SST-2",
        "prefix": None,
        "tau": 0.01,
        "set_predict_tau": 0.03,
        "y_scaler": 2,
        "target_root": DATA_ROOT + "/harsanyi_spilt/split/dataset=SST-2_model=bert-medium_epoch=20_bs=128_lr=0.01_seed=0_noise-label=False",
        "baseline_template": DATA_ROOT + "/harsanyi_spilt/split_baseline/dataset=SST-2_model=bert-medium_epoch=20_bs=128_lr=0.01_seed={seed}_noise-label=False",
    },
    {
        "key": "sst2-bert-base",
        "label": "BERT-Base on SST-2",
        "prefix": None,
        "tau": 0.01,
        "set_predict_tau": 0.03,
        "y_scaler": 2,
        "target_root": DATA_ROOT + "/harsanyi_spilt/split/dataset=SST-2_model=bert-base_epoch=10_bs=64_lr=0.01_seed=0_noise-label=False",
        "baseline_template": DATA_ROOT + "/harsanyi_spilt/split_baseline/dataset=SST-2_model=bert-base_epoch=10_bs=64_lr=0.01_seed={seed}_noise-label=False",
    },
]

LLM_MODELS = [
    {
        "key": "finance-qwen25-7b",
        "label": "Qwen2.5-7B on Finance",
        "tau": 0.03,
        "set_predict_tau": 0.08,
        "target": "Qwen__Qwen2.5-7B",
        "baselines": [
            "meta-llama__Meta-Llama-3-8B",
            "deepseek-ai__DeepSeek-R1-Distill-Llama-8B",
            "microsoft__phi-4",
            "google__gemma-3-12b-pt",
            "zai-org__GLM-Z1-9B-0414",
        ],
    },
    {
        "key": "finance-llama3-8b",
        "label": "Llama-3-8B on Finance",
        "tau": 0.03,
        "set_predict_tau": 0.05,
        "target": "meta-llama__Meta-Llama-3-8B",
        "baselines": [
            "Qwen__Qwen2.5-7B",
            "deepseek-ai__DeepSeek-R1-Distill-Llama-8B",
            "microsoft__phi-4",
            "google__gemma-3-12b-pt",
            "zai-org__GLM-Z1-9B-0414",
        ],
    },
    {
        "key": "finance-glmz1-9b",
        "label": "GLM-Z1-9B on Finance",
        "tau": 0.03,
        "set_predict_tau": 0.08,
        "target": "zai-org__GLM-Z1-9B-0414",
        "baselines": [
            "Qwen__Qwen2.5-7B",
            "meta-llama__Meta-Llama-3-8B",
            "deepseek-ai__DeepSeek-R1-Distill-Llama-8B",
            "microsoft__phi-4",
            "google__gemma-3-12b-pt",
        ],
    },
    {
        "key": "finance-phi4",
        "label": "Phi-4 on Finance",
        "tau": 0.03,
        "set_predict_tau": 0.05,
        "target": "microsoft__phi-4",
        "baselines": [
            "Qwen__Qwen2.5-7B",
            "meta-llama__Meta-Llama-3-8B",
            "deepseek-ai__DeepSeek-R1-Distill-Llama-8B",
            "google__gemma-3-12b-pt",
            "zai-org__GLM-Z1-9B-0414",
        ],
    },
]

def _llm_sample_names_with_single_cache(model_name):
    model_root = LLM_SINGLE_CACHE_ROOT / LLM_TASK / model_name
    names = set()
    for sample_dir in model_root.glob("Sample_*"):
        if not sample_dir.is_dir():
            continue
        model_dir = sample_dir / "model1"
        if (
            (model_dir / "I_and.npy").exists()
            and (model_dir / "I_or.npy").exists()
            and (model_dir / "before_sparsify" / "rewards.npy").exists()
        ):
            names.add(sample_dir.name)
    return names

def _llm_output_range(sample_dir):
    rewards = np.load(sample_dir / "model1" / "before_sparsify" / "rewards.npy")
    return abs(float(rewards[0] - rewards[-1]))

def _load_llm_interactions(model_name, sample_names, tau):
    model_root = LLM_SINGLE_CACHE_ROOT / LLM_TASK / model_name
    sample_dirs = [model_root / name for name in sample_names]
    I_and = np.asarray([np.load(sample_dir / "model1" / "I_and.npy") for sample_dir in sample_dirs], dtype=float)
    I_or = np.asarray([np.load(sample_dir / "model1" / "I_or.npy") for sample_dir in sample_dirs], dtype=float)
    masks = np.load(LLM_MASKS_PATH)
    for i, sample_dir in enumerate(sample_dirs):
        threshold = _llm_output_range(sample_dir) * tau
        I_and[i][np.abs(I_and[i]) < threshold] = 0.0
        I_or[i][np.abs(I_or[i]) < threshold] = 0.0
    return I_and, I_or, masks

def _compute_row_data(config):
    if "target" in config:
        model_names = [config["target"]] + list(config["baselines"])
        common_samples = None
        for model_name in model_names:
            sample_names = _llm_sample_names_with_single_cache(model_name)
            common_samples = sample_names if common_samples is None else common_samples & sample_names
        sample_names = sorted(common_samples, key=lambda name: int(name.replace("Sample_", "")))
        if not sample_names:
            raise RuntimeError(f"No common valid LLM samples for {config['label']}")

        target_I_and, target_I_or, masks = _load_llm_interactions(config["target"], sample_names, config["tau"])
        baseline_I_and_list = []
        baseline_I_or_list = []
        for baseline in config["baselines"][:SEED_NUM]:
            baseline_I_and, baseline_I_or, _ = _load_llm_interactions(baseline, sample_names, config["tau"])
            baseline_I_and_list.append(baseline_I_and)
            baseline_I_or_list.append(baseline_I_or)
        shared_I_and, shared_I_or = shared_interactions(
            target_I_and,
            target_I_or,
            baseline_I_and_list,
            baseline_I_or_list,
        )
    else:
        baseline_seeds = list(range(SEED_NUM))
        target_I_and, target_I_or, shared_I_and, shared_I_or, masks = load_target_and_shared_interactions(
            config["target_root"],
            config["baseline_template"],
            baseline_seeds,
            prefix=config["prefix"],
            tau=config["tau"],
        )

    all_pos, all_neg = aggregate_signed_orders(target_I_and, target_I_or, masks)
    gen_pos, gen_neg = aggregate_signed_orders(shared_I_and, shared_I_or, masks)

    predict_tau = config.get("set_predict_tau", config["tau"])
    cache_dir = OUTPUT_DIR / f"{config['key']}-compare_cache-predict_tau={predict_tau:g}"
    theory = disentangle_interaction_orders(
        target_I_and,
        target_I_or,
        masks,
        cache_dir=cache_dir,
        tau=config["tau"],
        set_predict_tau=config.get("set_predict_tau"),
        alpha_list=torch.linspace(0, 100, 101, dtype=torch.float64) * 1e-2,
        beta_list=torch.linspace(0, 100, 1001, dtype=torch.float64) * 1e-3,
        sigma2_list=torch.tensor([1e-3, 1e-2, 1e-1], dtype=torch.float64),
        gamma_list=torch.linspace(-1, 4, 1001, dtype=torch.float64) + 1,
    )

    return {
        "all_pos": all_pos,
        "all_neg": all_neg,
        "gen_pos": gen_pos,
        "gen_neg": gen_neg,
        "non_gen_pos": all_pos - gen_pos,
        "non_gen_neg": all_neg - gen_neg,
        "decay_pos": theory["decay_pos"],
        "decay_neg": -theory["decay_neg"],
        "spindle_pos": theory["spindle_pos"],
        "spindle_neg": -theory["spindle_neg"],
        "fit": theory,
    }


def _cache_roots(config):
    if "target" in config:
        models = [config["target"]] + list(config["baselines"][:SEED_NUM])
        return [LLM_MASKS_PATH] + [LLM_SINGLE_CACHE_ROOT / LLM_TASK / model for model in models]
    return [Path(config["target_root"])] + [
        Path(str(config["baseline_template"]).replace("{seed}", str(seed)))
        for seed in range(SEED_NUM)
    ]


def _load_row_data(config):
    signature = input_signature(config, _cache_roots(config))
    cache_path = OUTPUT_DIR / "row_data_cache" / f"{config['key']}.npz"
    cached = load_row_cache(cache_path, signature)
    if cached is not None:
        print(f"[cache hit] {config['key']}")
        return cached
    print(f"[cache miss] {config['key']}")
    data = _compute_row_data(config)
    save_row_cache(cache_path, signature, data)
    return data


def _finalize_axis(ax, n_dim, y_pos_max, y_neg_max):
    ax.set_xticks([1, n_dim] if n_dim > 4 else np.arange(1, n_dim + 1))
    ax.tick_params(pad=1)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True, nbins=3))
    ax.set_xlabel("Order $m$", fontsize=10)
    ax.xaxis.set_label_coords(0.5, -0.05)
    ax.set_ylim(y_neg_max, y_pos_max)

def _plot_row(axes, data, y_scaler=1):
    n_dim = len(data["all_pos"])

    y_pos_max = max(
        np.max(data["all_pos"]),
        np.max(data["gen_pos"]),
        np.max(data["decay_pos"]),
        np.max(data["non_gen_pos"]),
        np.max(data["spindle_pos"]),
    ) * 1.2
    y_neg_max = min(
        np.min(data["all_neg"]),
        np.min(data["gen_neg"]),
        np.min(data["decay_neg"]),
        np.min(data["non_gen_neg"]),
        np.min(data["spindle_neg"]),
    ) * 1.2
    if y_pos_max == 0:
        y_pos_max = 1.0
    if y_neg_max == 0:
        y_neg_max = -1.0

    non_gen_lower = (0.8 * y_neg_max - 0.2 * y_pos_max) / y_scaler
    non_gen_upper = (-0.2 * y_neg_max + 0.8 * y_pos_max) / y_scaler

    plot_signed_bars(axes[0], data["all_pos"], data["all_neg"], COLORS["all_pos"], COLORS["all_neg"], width=BAR_WIDTH, show_zero=False)
    plot_signed_bars(axes[1], data["gen_pos"], data["gen_neg"], COLORS["part_pos"], COLORS["part_neg"], width=BAR_WIDTH, show_zero=False)
    plot_signed_bars(axes[2], data["decay_pos"], data["decay_neg"], COLORS["part_pos"], COLORS["part_neg"], width=BAR_WIDTH, show_zero=False)
    plot_signed_bars(axes[3], data["non_gen_pos"], data["non_gen_neg"], COLORS["part_pos"], COLORS["part_neg"], width=BAR_WIDTH, show_zero=False)
    plot_signed_bars(axes[4], data["spindle_pos"], data["spindle_neg"], COLORS["part_pos"], COLORS["part_neg"], width=BAR_WIDTH, show_zero=False)

    for ax in axes[:3]:
        _finalize_axis(ax, n_dim, y_pos_max, y_neg_max)
    for ax in axes[3:]:
        _finalize_axis(ax, n_dim, non_gen_upper, non_gen_lower)

def compute_sim(data):
    """Compute the aggregate overlap between empirical and fitted components."""
    def _iu(A, B):
        A_abs = np.abs(A)
        B_abs = np.abs(B)
        return np.minimum(A_abs, B_abs).sum(), np.maximum(A_abs, B_abs).sum()

    pairs = [
        (data["gen_pos"], data["decay_pos"]),
        (data["gen_neg"], data["decay_neg"]),
        (data["non_gen_pos"], data["spindle_pos"]),
        (data["non_gen_neg"], data["spindle_neg"]),
    ]
    intersection, union = zip(*(_iu(A, B) for A, B in pairs))
    union_total = sum(union)
    return float(sum(intersection) / union_total) if union_total > 0 else 0.0

def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    apply_style(9)
    plt.rcParams.update({
        "font.family": "sans-serif",
        "font.size": 9,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "axes.edgecolor": "#333333",
        "xtick.major.size": 2.5,
        "ytick.major.size": 2.5,
        "ytick.major.width": 0.7,
        "grid.alpha": 0.3,
        "grid.linewidth": 0.5,
        "grid.color": "#CCCCCC",
        "figure.facecolor": "white",
        "axes.facecolor": "#F8F8F8",
    })

    models = MODELS + LLM_MODELS
    n_rows = len(models)
    figwidth = (
        LEFT_MARGIN
        + N_COLS * SUBPLOT_WIDTH
        + (N_COLS - 1 - 2) * WSPACE_INCHES
        + 2 * GROUP_GAP_INCHES
        + RIGHT_MARGIN
    )
    figheight = TOP_MARGIN + n_rows * SUBPLOT_HEIGHT + (n_rows - 1) * HSPACE_INCHES + BOTTOM_MARGIN
    fig = plt.figure(figsize=(figwidth, figheight))
    # Keep each axes background, while making only the outer canvas transparent.
    fig.patch.set_alpha(0.0)

    col_lefts = []
    x = LEFT_MARGIN
    for c in range(N_COLS):
        col_lefts.append(x)
        gap = GROUP_GAP_INCHES if c in (0, 2) else WSPACE_INCHES
        x += SUBPLOT_WIDTH + gap

    row_bottoms = []
    y = figheight - TOP_MARGIN - SUBPLOT_HEIGHT
    for _ in range(n_rows):
        row_bottoms.append(y)
        y -= SUBPLOT_HEIGHT + HSPACE_INCHES

    axes = []
    for r in range(n_rows):
        row = []
        for c in range(N_COLS):
            row.append(fig.add_axes([
                col_lefts[c] / figwidth,
                row_bottoms[r] / figheight,
                SUBPLOT_WIDTH / figwidth,
                SUBPLOT_HEIGHT / figheight,
            ]))
        axes.append(row)

    sim_results = []
    for r, config in enumerate(models):
        print(f"[plot] {config['label']}")
        data = _load_row_data(config)
        fit = data["fit"]
        sim = compute_sim(data)
        print(
            "[fit]",
            config["key"],
            f"alpha={fit['alpha']:.4g}",
            f"beta={fit['beta']:.4g}",
            f"sigma2={fit['sigma2']:.4g}",
            f"gamma={fit['gamma']:.4g}",
            f"loss={fit['min_loss']:.4g}",
            f"SIM={sim:.4f}",
        )
        sim_results.append((config["key"], config["label"], sim))
        _plot_row(axes[r], data, y_scaler=config.get("y_scaler", 1))
        axes[r][4].text(
            1.10, 0.5, rf"$\mathit{{SIM}} = {sim:.3f}$",
            transform=axes[r][4].transAxes, ha="left", va="center",
            fontsize=13, fontfamily="DejaVu Serif", clip_on=False,
        )

    pdf = OUTPUT_DIR / "main.pdf"
    png = OUTPUT_DIR / "main.png"
    fig.savefig(pdf, bbox_inches=None, pad_inches=0.0, transparent=False)
    fig.savefig(png, bbox_inches=None, pad_inches=0.0, dpi=300, transparent=False)
    plt.close(fig)
    sim_csv = OUTPUT_DIR / "sim_results.csv"
    np.savetxt(
        sim_csv,
        np.asarray(sim_results, dtype=object),
        delimiter=",",
        fmt="%s",
        header="key,label,SIM",
        comments="",
    )
    print("\n[SIM summary]")
    for key, label, sim in sim_results:
        print(f"{key:24s} SIM={sim:.6f}  {label}")
    print(f"[done] {pdf}")
    print(f"[done] {png}")
    print(f"[done] {sim_csv}")
    return sim_results

In [5]:
def compute_sim(data):
    """Compute Similarity Index Metric (SIM) from decomposed order-wise interactions.

    SIM = (sum of min overlaps) / (sum of max unions)
    for (generalizable, decay) and (non-generalizable, spindle) respectively.
    """

    def _iu(A, B):
        A_abs = np.abs(A)
        B_abs = np.abs(B)
        return np.minimum(A_abs, B_abs).sum(), np.maximum(A_abs, B_abs).sum()

    I_gen_decay_pos, U_gen_decay_pos = _iu(data["gen_pos"], data["decay_pos"])
    I_gen_decay_neg, U_gen_decay_neg = _iu(data["gen_neg"], data["decay_neg"])
    I_non_spindle_pos, U_non_spindle_pos = _iu(
        data["non_gen_pos"], data["spindle_pos"]
    )
    I_non_spindle_neg, U_non_spindle_neg = _iu(
        data["non_gen_neg"], data["spindle_neg"]
    )

    I_total = (
        I_gen_decay_pos
        + I_gen_decay_neg
        + I_non_spindle_pos
        + I_non_spindle_neg
    )
    U_total = (
        U_gen_decay_pos
        + U_gen_decay_neg
        + U_non_spindle_pos
        + U_non_spindle_neg
    )
    return float(I_total / U_total) if U_total > 0 else 0.0

## Run


In [6]:
# Generate the combined figure with image/NLP models and LLM finance rows.
# Outputs:
#   saved-results/main.pdf
#   saved-results/main.png
main()


[plot] ResNet-20 on MNIST
[cache miss] mnist-resnet20


[fit] mnist-resnet20 alpha=0.79 beta=0.0458 sigma2=0.001 gamma=1.105 loss=0.8698 SIM=0.7552
[plot] ResNet-20 on CIFAR-10
[cache miss] cifar10-resnet20


[fit] cifar10-resnet20 alpha=0.82 beta=0.0045 sigma2=0.001 gamma=1.255 loss=0.2858 SIM=0.8201
[plot] VGG-11 on CIFAR-10


[cache miss] cifar10-vgg11


[fit] cifar10-vgg11 alpha=0.75 beta=0.0041 sigma2=0.001 gamma=1.24 loss=0.1724 SIM=0.7963
[plot] BERT-Medium on SST-2


...

[fit] finance-llama3-8b alpha=0.88 beta=0.01 sigma2=0.001 gamma=1.175 loss=0.9017 SIM=0.7987
[plot] GLM-Z1-9B on Finance
[cache miss] finance-glmz1-9b


[fit] finance-glmz1-9b alpha=0.78 beta=0.0158 sigma2=0.001 gamma=1.26 loss=0.4 SIM=0.8741
[plot] Phi-4 on Finance
[cache miss] finance-phi4


[fit] finance-phi4 alpha=0.77 beta=0.0163 sigma2=0.001 gamma=1.145 loss=0.6882 SIM=0.7530



[SIM summary]
mnist-resnet20           SIM=0.755211  ResNet-20 on MNIST
cifar10-resnet20         SIM=0.820103  ResNet-20 on CIFAR-10
cifar10-vgg11            SIM=0.796338  VGG-11 on CIFAR-10
sst2-bert                SIM=0.850941  BERT-Medium on SST-2
sst2-bert-base           SIM=0.862123  BERT-Base on SST-2
finance-qwen25-7b        SIM=0.804918  Qwen2.5-7B on Finance
finance-llama3-8b        SIM=0.798665  Llama-3-8B on Finance
finance-glmz1-9b         SIM=0.874061  GLM-Z1-9B on Finance
finance-phi4             SIM=0.753031  Phi-4 on Finance
[done] <package>/visualization/Fig_06_fit/outputs/main.pdf
[done] <package>/visualization/Fig_06_fit/outputs/main.png
[done] <package>/visualization/Fig_06_fit/outputs/sim_results.csv


[('mnist-resnet20', 'ResNet-20 on MNIST', 0.7552112946798971),
 ('cifar10-resnet20', 'ResNet-20 on CIFAR-10', 0.8201028730725828),
 ('cifar10-vgg11', 'VGG-11 on CIFAR-10', 0.7963377462215667),
 ('sst2-bert', 'BERT-Medium on SST-2', 0.8509407065187748),
 ('sst2-bert-base', 'BERT-Base on SST-2', 0.8621234786004316),
 ('finance-qwen25-7b', 'Qwen2.5-7B on Finance', 0.8049176597254138),
 ('finance-llama3-8b', 'Llama-3-8B on Finance', 0.7986653299850848),
 ('finance-glmz1-9b', 'GLM-Z1-9B on Finance', 0.874060626984594),
 ('finance-phi4', 'Phi-4 on Finance', 0.7530314174874402)]